In [1]:
import os
import importlib

In [2]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  IN_COLAB = True
except:
  IN_COLAB = False

Mounted at /content/drive


In [3]:
if IN_COLAB:
    !pip install -q "peft==0.13.2" "transformers==4.50.0" "lm_eval==0.4.1" "datasets==2.18.0"
    # !pip install -q "peft==0.13.2" "transformers==4.57.3" "lm_eval==0.4.9.2" "datasets==2.18.0"
    # !pip install -q "lm_eval[hf]"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 138.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not

In [4]:
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

if IN_COLAB:
    os.makedirs(LOG_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [5]:
# import lm_eval
# importlib.reload(slicegpt)
# importlib.reload(sliced_rag)
# importlib.reload(lm_eval)

In [6]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

In [7]:
# model_id = "google/gemma-3-270m-it"
# model_id = "google/gemma-3-1b-it"

In [8]:
# def load_model(model_id):
#   model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
#   tokenizer = AutoTokenizer.from_pretrained(model_id)
#   return model, tokenizer

In [9]:
# model, tokenizer = load_model(model_id)

In [10]:
# def generate_text(prompt):
#   model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#   generated_ids = model.generate(**model_inputs, max_new_tokens=50, do_sample=False)
#   generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
#   return generated_text

In [11]:
# prompt = """
# Question: What is the capital of France?

# Answer: """

In [12]:
# print(generate_text(prompt))

In [13]:
# from sliced_rag.evaluation import eval_baseline


In [14]:
# model = "google/gemma-3-270m-it"
# model = "google/gemma-3-1b-it"
model = "google/flan-t5-base"
model_name = model.replace("/", "-")
# sparsities = [0.0, 0.10, 0.25, 0.40, 0.60]

# Evaluation

Running Gemma3 eval separately because of differences in the HF transformers library versions (Gemma3 requires transformers>=4.50, while slicegpt (and therefore sliced_rag) requires transformers==4.41.0)

In [20]:
import json
import logging
import sys
# import os

import lm_eval
from lm_eval import tasks
from lm_eval import utils as lm_eval_utils
from lm_eval.api.registry import ALL_TASKS
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import initialize_tasks

# from slicegpt import gpu_utils, hf_utils, utils
# from slicegpt.config import config

# import os
# import json
# import torch
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import evaluate
# from tqdm import tqdm

initialize_tasks()

logging.basicConfig(
    level=logging.DEBUG,
    stream=sys.stdout,
    format="%(levelname)s: %(message)s"
)

def get_logger():
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)

    # remove handlers if already set, to avoid double logging
    for h in list(logger.handlers):
        logger.removeHandler(h)

    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    formatter = logging.Formatter('%(levelname)s - %(message)s')
    handler.setFormatter(formatter)

    logger.addHandler(handler)

    return logger

def eval_baseline(args):
    """
    Run LM Evaluation Harness on either:
      - a sliced (pruned) model, if 'sliced_model_path' is provided
      - or a dense HF model, if not.
    """
    logger = get_logger()
    logger.info("Running Evaluation")

    # ---------- DENSE BASELINE BRANCH ----------
    logger.info(
        f"Loading DENSE baseline model {args['model']} directly from HF hub"
    )

    # Here we let LM Eval Harness load the model & tokenizer itself
    hflm = HFLM(
        pretrained=args["model"],
        batch_size=args["batch_size"],
        dtype="float16"
    )
    hflm.model.config.use_cache=False

    # ---------- Task selection ----------
    if args["tasks"] is None:
        logger.warning(
            "args['tasks'] is None -> using ALL_TASKS. "
            "This may be very slow / memory-heavy."
        )
        task_names = tasks.ALL_TASKS
    else:
        task_names = args["tasks"]

    logger.info(f"Selected Tasks: {task_names}")

    # ---------- Run evaluation ----------
    results = lm_eval.simple_evaluate(
        hflm,
        tasks=task_names,
        num_fewshot=args["num_fewshot"],
        batch_size=args["batch_size"],
        limit=args["limit"],
        write_out=True,
        log_samples=args["log_samples"]
    )

    logger.info("Results (metrics only):")
    # logger.info(results["results"])
    logger.info(results)


    # ---------- Save results ----------
    os.makedirs(args["save_dir"], exist_ok=True)

    # Use sparsity=0.0 if not present, so filenames still make sense
    sparsity_val = float(args.get("sparsity", 0.0))
    sparsity_tag = f"{sparsity_val:.2f}"

    result_path = os.path.join(
        args["save_dir"],
        f"results_s{sparsity_tag}_{'_'.join(task_names)}_light.json",
    )

    with open(result_path, "w") as f:
        try:
          json.dump(results, f, indent=2, skipkeys=True)
        except:
          json.dump(results["results"], f, indent=2, skipkeys=True)

    logger.info(f"Saved results to {result_path}")

    return results


  0%|          | 0/11873 [00:38<?, ?it/s]


WARNING - Some tasks could not be loaded due to missing dependencies. Run with `--verbosity DEBUG` for full details.
WARNING - Some tasks could not be loaded due to missing dependencies. Run with `--verbosity DEBUG` for full details.


In [23]:
EVAL_DIR = os.path.join(BASE_EVAL_DIR, f"{model_name}_unpruned")

args = {
    "model": model,
    "sliced_model_path": None,
    "sparsity": 0.0,
    "save_dir": EVAL_DIR,
    "tasks": ["coqa"],
    "num_fewshot": 0,
    "batch_size": 32,
    "round_interval": 8,
    "limit": None,
    "log_samples": False
}

results = eval_baseline(args)

INFO - Running Evaluation
INFO - Loading DENSE baseline model google/flan-t5-base directly from HF hub
INFO - Using device 'cuda'
INFO - Selected Tasks: ['coqa']


INFO - num_fewshot has been set to 0 for coqa in its config. Manual configuration will be ignored.
INFO - Building contexts for task on rank 0...
INFO - Task: coqa; document 0; context prompt (starting on next line):
Once upon a time, in a barn near a farm house, there lived a little white kitten named Cotton. Cotton lived high up in a nice warm place above the barn where all of the farmer's horses slept. But Cotton wasn't alone in her little home above the barn, oh no. She shared her hay bed with her mommy and 5 other sisters. All of her sisters were cute and fluffy, like Cotton. But she was the only white one in the bunch. The rest of her sisters were all orange with beautiful white tiger stripes like Cotton's mommy. Being different made Cotton quite sad. She often wished she looked like the rest of her family. So one day, when Cotton found a can of the old farmer's orange paint, she used it to paint herself like them. When her mommy and sisters found her they started laughing. 

"Wh

100%|██████████| 500/500 [00:20<00:00, 24.22it/s]


INFO - Results (metrics only):
INFO - {'results': {'coqa': {'em,none': 0.437, 'em_stderr,none': 0.020253783964066057, 'f1,none': 0.5329550618400823, 'f1_stderr,none': 0.01973267818196054, 'alias': 'coqa'}}, 'configs': {'coqa': {'task': 'coqa', 'dataset_path': 'EleutherAI/coqa', 'training_split': 'train', 'validation_split': 'validation', 'doc_to_text': 'def doc_to_text(doc):\n    # Given a passage p, the conversation history {q1, a1, . . . qi−1, ai−1}\n    # and a question qi, the task is to predict the answer ai\n    doc_text = doc["story"] + "\\n\\n"\n    for q, a in zip_longest(\n        doc["questions"]["input_text"], doc["answers"]["input_text"][:-1]\n    ):  # omit target answer ai\n        question = f"Q: {q}\\n\\n"\n        answer = f"A: {a}\\n\\n" if a is not None else "A:"\n        doc_text += question + answer\n    return doc_text\n', 'doc_to_target': 'def doc_to_target(doc):\n    turn_id = len(doc["questions"]["input_text"])\n    # Returns unique answers and valid alternati

In [24]:
results

{'results': {'coqa': {'em,none': 0.437,
   'em_stderr,none': 0.020253783964066057,
   'f1,none': 0.5329550618400823,
   'f1_stderr,none': 0.01973267818196054,
   'alias': 'coqa'}},
 'configs': {'coqa': {'task': 'coqa',
   'dataset_path': 'EleutherAI/coqa',
   'training_split': 'train',
   'validation_split': 'validation',
   'doc_to_text': 'def doc_to_text(doc):\n    # Given a passage p, the conversation history {q1, a1, . . . qi−1, ai−1}\n    # and a question qi, the task is to predict the answer ai\n    doc_text = doc["story"] + "\\n\\n"\n    for q, a in zip_longest(\n        doc["questions"]["input_text"], doc["answers"]["input_text"][:-1]\n    ):  # omit target answer ai\n        question = f"Q: {q}\\n\\n"\n        answer = f"A: {a}\\n\\n" if a is not None else "A:"\n        doc_text += question + answer\n    return doc_text\n',
   'doc_to_target': 'def doc_to_target(doc):\n    turn_id = len(doc["questions"]["input_text"])\n    # Returns unique answers and valid alternatives (Some 